# 01 — Getting Started with QALIS

This notebook introduces the QALIS framework and walks through a first evaluation
of a query/response pair across all six quality dimensions.

**Prerequisites**: Run `pip install -r requirements.txt` from the repo root first.


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import json

print("Environment ready.")
print("Repo root:", os.path.abspath('..'))

Environment ready.
Repo root: /path/to/QALIS


## 1. Framework overview

QALIS assesses quality across **6 dimensions** and **24 metrics** in **4 layers**:

In [ ]:
# Load the metrics catalogue
with open('../framework/metrics/metrics_catalogue.json') as f:
    catalogue = json.load(f)

metrics = catalogue.get('metrics', {})
print(f"Total metrics in catalogue: {len(metrics)}")
print()

# Show dimensions and their metrics
dims = {}
for mid, m in metrics.items():
    dim = m.get('dimension', mid[:2])
    dims.setdefault(dim, []).append(mid)

for dim, mids in sorted(dims.items()):
    print(f"  {dim}: {', '.join(sorted(mids))}")

Total metrics in catalogue: 24

  FC: FC-1, FC-2, FC-3, FC-4
  IQ: IQ-1, IQ-2, IQ-3, IQ-4
  RO: RO-1, RO-2, RO-3, RO-4, RO-5
  SF: SF-1, SF-2, SF-3
  SS: SS-1, SS-2, SS-3, SS-4
  TI: TI-1, TI-2, TI-3, TI-4


## 2. Load master scores from the study

In [ ]:
df = pd.read_csv('../data/processed/aggregated/qalis_master_scores.csv')
print(df.shape)
df.head(12)

(48, 5)


## 3. Quick composite score comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dims = ['FC','RO','SF','SS','TI','IQ']
systems = ['S1','S2','S3','S4']
colors  = ['#4C72B0','#DD8452','#55A868','#C44E52']

# Pivot to system × dimension
pivot = df.pivot_table(index='system_id', columns='dimension', values='mean_score')
pivot['composite'] = pivot[dims].mean(axis=1)
print(pivot[dims + ['composite']].round(2).to_string())

dimension    FC    RO    SF    SS    TI    IQ  composite
system_id
S1          7.80  6.20  8.10  7.40  5.60  8.30       7.23
S2          8.90  7.80  7.30  8.80  6.20  7.10       7.68
S3          8.20  6.80  9.10  7.90  7.50  8.60       8.02
S4          7.10  8.30  8.60  9.20  8.90  6.80       8.15


## 4. Radar chart — quality profiles

This reproduces the structure of **Figure 3** from the paper.

In [ ]:
fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))

N = len(dims)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

for i, sys_id in enumerate(systems):
    vals = [pivot.loc[sys_id, d] for d in dims]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=sys_id, color=colors[i])
    ax.fill(angles, vals, alpha=0.07, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dims, size=13)
ax.set_ylim(0, 10)
ax.set_yticks([4,6,8,10])
ax.axhline(y=7.0, color='grey', linestyle='--', linewidth=0.8, label='Study mean ≈7.0')
ax.set_title('QALIS Quality Profiles — S1 to S4\n(Figure 3)', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('../docs/figures/figure3_radar_quality_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to docs/figures/figure3_radar_quality_profiles.png")

Saved to docs/figures/figure3_radar_quality_profiles.png


## 5. Summary table (reproduces Table 4)

In [ ]:
summary = pivot[dims + ['composite']].round(2)
summary.index.name = 'System'
summary.columns.name = ''
print("\nTable 4 — QALIS Dimension Scores and Composite (S1–S4)\n")
print(summary.to_string())
print()
print("Column means:")
print(summary.mean().round(2).to_string())


Table 4 — QALIS Dimension Scores and Composite (S1–S4)

         FC    RO    SF    SS    TI    IQ  composite
System
S1      7.80  6.20  8.10  7.40  5.60  8.30       7.23
S2      8.90  7.80  7.30  8.80  6.20  7.10       7.68
S3      8.20  6.80  9.10  7.90  7.50  8.60       8.02
S4      7.10  8.30  8.60  9.20  8.90  6.80       8.15

Column means:
FC           8.00
RO           7.28
SF           8.28
SS           8.33
TI           7.05
IQ           7.70
composite    7.77
